# Phase 3 — Full Ablation Sweep (from scratch)

**Run all cells top-to-bottom on a fresh runtime.**

| Cell | Content | Time |
|------|---------|------|
| P1 | Clone/pull repo | <1 min |
| P2 | Install deps + disable wandb | 2 min |
| P3 | Mount Drive + copy data | 1 min |
| P4 | Generate thesis diagrams (fig01–fig11) | 10 sec |
| P5 | Sanity check — baseline on 5 frames | ~3 min |
| P6 | Fast sweep — baseline + 3A + 3B + 3D (4 configs × 50 frames) | ~2.5 h |
| P6b | **Optional** — restart (3C) + combined (~6 h extra) | optional |
| P7 | Results table + Pareto curve | 1 min |
| P8 | Save everything to Drive | 2 min |

**Phase 3 configs:**
- `phase3_baseline` — same as Phase 2 best (reference)
- `phase3_objectness` — 3A: objectness-aware loss weighting
- `phase3_momentum` — 3B: MI-Adam gradient momentum (μ=0.9)
- `phase3_ssim` — 3D: combined LPIPS + SSIM constraint
- `phase3_restart` — 3C: multi-restart (n=3) — **expensive, ~3× compute per frame**
- `phase3_combined` — all 4 improvements — **most expensive**


## Cell P1 — Clone / pull repo

In [ ]:
import os

REPO_DIR = '/content/pfe-latent-attack'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/03aziz03/pfe-latent-attack.git {REPO_DIR}
else:
    print('Repo already present — pulling latest ...')
    !git -C {REPO_DIR} pull origin main

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

# Verify all critical files are present
required = [
    'scripts/run_phase3_ablation.py',
    'scripts/generate_thesis_figs.py',
    'src/eval/metrics.py',
    'src/losses.py',
    'src/attack.py',
    'configs/phase3_baseline.yaml',
    'configs/phase3_objectness.yaml',
    'configs/phase3_momentum.yaml',
    'configs/phase3_restart.yaml',
    'configs/phase3_ssim.yaml',
    'configs/phase3_combined.yaml',
]
all_ok = True
for f in required:
    ok = os.path.exists(f)
    all_ok = all_ok and ok
    print(f"  {'✓' if ok else '✗ MISSING':<12} {f}")
if not all_ok:
    raise RuntimeError("Missing files — check git pull output above")
print('\n✓ All files present')


## Cell P2 — Install dependencies + disable wandb

In [ ]:
# Install deps (no numpy pin — let Colab decide)
%pip install -q "ultralytics>=8.2,<9" lpips>=0.1.4 "torchmetrics>=1.0" pyyaml pillow tqdm

# Disable wandb before ultralytics imports it
import os, sys
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE']     = 'disabled'

try:
    from ultralytics.utils import SETTINGS
    SETTINGS['wandb'] = False
except Exception:
    pass

for mod in list(sys.modules):
    if mod == 'wandb' or mod.startswith('wandb.'):
        del sys.modules[mod]

print('✓ wandb disabled')

# Sanity check versions
import torch, numpy as np
print(f'numpy   {np.__version__}')
print(f'torch   {torch.__version__}  cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     {torch.cuda.get_device_name(0)}')
import ultralytics
print(f'ultralytics {ultralytics.__version__}')

try:
    from torchmetrics.detection import MeanAveragePrecision
    print('✓ torchmetrics.detection available')
except ImportError:
    print('✗ torchmetrics missing — mAP_drop will be skipped')

try:
    import lpips
    print('✓ lpips available')
except ImportError:
    print('✗ lpips missing — LPIPS will be NaN')


## Cell P3 — Mount Drive + copy data

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

# Drive layout (from training session):
#   MyDrive/PFE/runs/yolov8n_detrac/weights/best.pt
#   MyDrive/PFE/images_50/           (50 DETRAC frames)
DRIVE_PFE = '/content/drive/MyDrive/PFE'

# ── copy detector weights ────────────────────────────────────────────────────
os.makedirs('runs/yolov8n_detrac/weights', exist_ok=True)
best_local = 'runs/yolov8n_detrac/best.pt'
if not os.path.exists(best_local):
    # try both common Drive locations
    candidates = [
        f'{DRIVE_PFE}/runs/yolov8n_detrac/weights/best.pt',
        f'{DRIVE_PFE}/runs/yolov8n_detrac/best.pt',
        f'{DRIVE_PFE}/best.pt',
    ]
    for src in candidates:
        if os.path.exists(src):
            shutil.copy(src, best_local)
            print(f'✓ best.pt copied from {src}')
            break
    else:
        raise FileNotFoundError(
            f'best.pt not found in Drive. Checked: {candidates}')
else:
    print(f'✓ best.pt already present ({os.path.getsize(best_local)//1024} KB)')

# ── copy 50 frames ───────────────────────────────────────────────────────────
if not os.path.exists('data/images_50'):
    candidates = [
        f'{DRIVE_PFE}/images_50',
        f'{DRIVE_PFE}/data/images_50',
    ]
    for src in candidates:
        if os.path.exists(src):
            shutil.copytree(src, 'data/images_50')
            print(f'✓ images_50 copied from {src}')
            break
    else:
        raise FileNotFoundError(f'images_50 not found in Drive. Checked: {candidates}')
else:
    n = len([f for f in os.listdir('data/images_50') if f.endswith('.jpg')])
    print(f'✓ data/images_50 already present ({n} frames)')

n50 = len([f for f in os.listdir('data/images_50') if f.endswith('.jpg')])
print(f'\n✓ Ready: {n50} frames, best.pt loaded')

os.makedirs('results/phase3', exist_ok=True)
os.makedirs('results/phase3_sanity', exist_ok=True)
os.makedirs('figures', exist_ok=True)


## Cell P4 — Generate thesis diagram figures (fig01–fig11)

Runs entirely on CPU with matplotlib — no GPU needed (~10 seconds).
Generates `figures/fig01_*.pdf/.png` through `figures/fig11_*.pdf/.png`.


In [ ]:
!python scripts/generate_thesis_figs.py
import os
figs = sorted(os.listdir('figures'))
print(f'\n{len(figs)} files in figures/: {figs[:6]} ...')


## Cell P5 — Sanity check: baseline on 5 frames (~3 min)

Verifies the full pipeline (attack → detect → metrics → image save) works
before committing to the 3-hour full sweep.

**Expected results on 5 frames:**
- DFR_prop  ≈ +0.50  (50% detections removed on average)
- ASR       ≈  0.60  (3 of 5 frames fully suppressed)
- LPIPS     ≈  0.09  (very low distortion)


In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python scripts/run_phase3_ablation.py \
    --data      data/images_50 \
    --n_frames  5 \
    --output    results/phase3_sanity \
    --configs   configs/phase3_baseline.yaml \
    --no_resume \
    --save_images


In [ ]:
import json

s = json.load(open('results/phase3_sanity/phase3_baseline/summary.json'))
print('Sanity check results (5 frames):')
print(f"  DFR_prop    = {s['mean_dfr']:+.4f}   (expected ≈ +0.50)")
print(f"  DFR_strict  = {s['dfr_strict_rate']:.3f}    (expected ≈ 0.20)")
print(f"  ASR         = {s['mean_asr']:.3f}    (expected ≈ 0.60)")
print(f"  mAP_drop    = {s.get('mean_map_drop', float('nan')):.4f}")
print(f"  LPIPS       = {s['mean_lpips']:.4f}   (expected ≈ 0.09)")
print()

# Check images were saved
from pathlib import Path
imgs = list(Path('results/phase3_sanity/phase3_baseline/images').glob('*.jpg'))
print(f'✓ {len(imgs)} images saved (expected 20 = 5 frames × 4 types)')
if len(imgs) > 0:
    print('  Types:', set(p.stem.split('_')[-1] for p in imgs))


## Cell P6 — Fast sweep — 4 configs × 50 frames (~2.5 h)

Runs baseline + objectness (3A) + momentum (3B) + LPIPS+SSIM (3D).
Excludes restart and combined to keep total time under 3 h.

- `--resume` lets you restart if the session disconnects
- `--save_images` saves clean / adv / compare / diff JPEGs per frame


In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python scripts/run_phase3_ablation.py \
    --data        data/images_50 \
    --n_frames    50 \
    --output      results/phase3 \
    --configs     configs/phase3_baseline.yaml \
                  configs/phase3_objectness.yaml \
                  configs/phase3_momentum.yaml \
                  configs/phase3_ssim.yaml \
    --resume \
    --save_images


## Cell P6b — Optional: restart (3C) + combined (~6 h)

Run only if you have spare Colab Pro time after P6 finishes.
`phase3_restart` runs 3 random initialisations per frame (~90 min).
`phase3_combined` stacks all 4 improvements (~90 min).

Results will merge into `results/phase3/` alongside P6 outputs.


In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Uncomment to run — expensive!
# !python scripts/run_phase3_ablation.py \
#     --data        data/images_50 \
#     --n_frames    50 \
#     --output      results/phase3 \
#     --configs     configs/phase3_restart.yaml \
#                   configs/phase3_combined.yaml \
#     --resume \
#     --save_images

print('Cell P6b skipped.')
print('Uncomment the lines above to run restart + combined configs.')


## Cell P7 — Results table + Pareto curve

In [ ]:
import json
from pathlib import Path

results_dir = Path('results/phase3')
summaries = {}
for cfg_dir in sorted(results_dir.iterdir()):
    sf = cfg_dir / 'summary.json'
    if sf.exists():
        summaries[cfg_dir.name] = json.load(open(sf))

if not summaries:
    print('No results yet — run Cell P6 first')
else:
    header = f"{'Config':<28} {'DFR':>8} {'DFR_str':>9} {'ASR':>7} {'mAP_drop':>10} {'LPIPS':>8} {'N':>5}"
    print(header)
    print('-' * len(header))
    for tag, s in summaries.items():
        def fmt(k, w=8, plus=False):
            v = s.get(k)
            if v is None or v != v: return ' ' * w + 'N/A'
            return f"{v:+{w}.4f}" if plus else f"{v:{w}.4f}"
        print(f"{tag:<28} {fmt('mean_dfr', plus=True)} {fmt('dfr_strict_rate',9)} "
              f"{fmt('mean_asr',7)} {fmt('mean_map_drop',10)} {fmt('mean_lpips',8)} "
              f"{s.get('n_frames',0):>5}")


In [ ]:
import json, math
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = Path('results/phase3')
summaries = {}
for cfg_dir in sorted(results_dir.iterdir()):
    sf = cfg_dir / 'summary.json'
    if sf.exists():
        summaries[cfg_dir.name] = json.load(open(sf))

COLORS = {
    'phase3_baseline':   '#2196F3',
    'phase3_objectness': '#4CAF50',
    'phase3_momentum':   '#FF9800',
    'phase3_ssim':       '#9C27B0',
    'phase3_restart':    '#F44336',
    'phase3_combined':   '#111111',
}
LABELS = {
    'phase3_baseline':   'Baseline',
    'phase3_objectness': '+ Objectness (3A)',
    'phase3_momentum':   '+ MI-Adam (3B)',
    'phase3_ssim':       '+ LPIPS+SSIM (3D)',
    'phase3_restart':    '+ Restart (3C)',
    'phase3_combined':   'Combined',
}

fig, ax = plt.subplots(figsize=(8, 6))
for tag, s in summaries.items():
    lv = s.get('mean_lpips');  dv = s.get('mean_dfr')
    if lv is None or dv is None: continue
    if not (math.isfinite(lv) and math.isfinite(dv)): continue
    c = COLORS.get(tag, '#888'); lbl = LABELS.get(tag, tag)
    ax.scatter(lv, dv, s=160, color=c, zorder=5, label=lbl)
    ax.annotate(lbl, (lv, dv), textcoords='offset points',
                xytext=(6, 4), fontsize=8, color=c)

ax.set_xlabel('LPIPS (lower = less distortion)', fontsize=11)
ax.set_ylabel('DFR_prop (higher = more effective)', fontsize=11)
ax.set_title('Phase 3 Pareto: Effectiveness vs. Perceptual Quality', fontsize=12)
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/phase3/phase3_pareto.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved → results/phase3/phase3_pareto.png')


## Cell P8 — Save all results + images + figures to Drive

In [ ]:
import shutil, os, time
from pathlib import Path

DRIVE_PFE = '/content/drive/MyDrive/PFE'

def safe_copy(src, dst):
    src, dst = Path(src), Path(dst)
    if not src.exists():
        print(f'  ⚠  {src} does not exist — skipping')
        return
    if dst.exists():
        backup = str(dst) + f'_backup_{int(time.time())}'
        shutil.move(str(dst), backup)
        print(f'  ↩  Backed up existing {dst.name}')
    shutil.copytree(str(src), str(dst))
    n = sum(1 for _ in dst.rglob('*') if _.is_file())
    print(f'  ✓  {src} → Drive  ({n} files)')

print('Saving to Drive...')
safe_copy('results/phase3',  f'{DRIVE_PFE}/results/phase3')
safe_copy('results/phase3_sanity', f'{DRIVE_PFE}/results/phase3_sanity')
safe_copy('figures',         f'{DRIVE_PFE}/figures')

# Also copy the Pareto PNG separately for quick access
pareto = Path('results/phase3/phase3_pareto.png')
if pareto.exists():
    shutil.copy(str(pareto), f'{DRIVE_PFE}/phase3_pareto.png')
    print(f'  ✓  Pareto PNG → Drive/PFE/phase3_pareto.png')

print('\n✓ All done. Check MyDrive/PFE/ in Google Drive.')
print('  results/phase3/phase3_summary.json  ← paste this to Cowork session')
print('  results/phase3/phase3_table.md      ← paste this to Cowork session')
